In [1]:
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Display first 5 row

In [2]:
file = "dataset/student_module_pass_chance_regression.csv"
df = pd.read_csv(file)
df.head()

,Student_ID,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week,Pass_Chance_Percentage
0,ST0001,Female,91.8,79.0,77.5,75.1,7.2,69.8,10.0,2,8.8,76.82
1,ST0002,Male,64.7,44.9,81.3,54.1,7.1,58.8,1.8,5,9.3,35.13
2,ST0003,Female,34.9,37.4,61.0,31.0,13.0,51.8,1.4,6,10.2,25.48
3,ST0004,Female,49.5,44.6,33.6,44.1,1.0,78.1,4.8,4,4.1,32.98
4,ST0005,Male,97.3,92.5,100.0,100.0,4.2,100.0,10.0,1,7.2,91.71


In [3]:
encode = LabelEncoder()
df['Gender'] = encode.fit_transform(df['Gender'])

In [4]:
## after label encode

In [5]:
df.head()

,Student_ID,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week,Pass_Chance_Percentage
0,ST0001,0,91.8,79.0,77.5,75.1,7.2,69.8,10.0,2,8.8,76.82
1,ST0002,1,64.7,44.9,81.3,54.1,7.1,58.8,1.8,5,9.3,35.13
2,ST0003,0,34.9,37.4,61.0,31.0,13.0,51.8,1.4,6,10.2,25.48
3,ST0004,0,49.5,44.6,33.6,44.1,1.0,78.1,4.8,4,4.1,32.98
4,ST0005,1,97.3,92.5,100.0,100.0,4.2,100.0,10.0,1,7.2,91.71


In [6]:
df = df.drop(['Student_ID'], axis=1)

## After drop

In [7]:
df.head()

,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week,Pass_Chance_Percentage
0,0,91.8,79.0,77.5,75.1,7.2,69.8,10.0,2,8.8,76.82
1,1,64.7,44.9,81.3,54.1,7.1,58.8,1.8,5,9.3,35.13
2,0,34.9,37.4,61.0,31.0,13.0,51.8,1.4,6,10.2,25.48
3,0,49.5,44.6,33.6,44.1,1.0,78.1,4.8,4,4.1,32.98
4,1,97.3,92.5,100.0,100.0,4.2,100.0,10.0,1,7.2,91.71


## show issing values

In [8]:
df.isna().sum()

Gender                          0
Attendance_Percentage          45
Assignment_Average             45
Quiz_Average                   45
Midterm_Marks                   0
Study_Hours_Per_Week           45
Previous_Module_Marks          45
Class_Participation_Score       0
Missed_Assignments              0
LMS_Activity_Hours_Per_Week    45
Pass_Chance_Percentage          0
dtype: int64

## DETECT OUTLIERS USING IQR

In [9]:
# Select numerical columns
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns

# Detect outliers using IQR
for column in numerical_columns:
    
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]
    
    print(f"{column}: {len(outliers)} outliers")

Gender: 0 outliers
Attendance_Percentage: 15 outliers
Assignment_Average: 16 outliers
Quiz_Average: 14 outliers
Midterm_Marks: 8 outliers
Study_Hours_Per_Week: 76 outliers
Previous_Module_Marks: 11 outliers
Class_Participation_Score: 0 outliers
Missed_Assignments: 24 outliers
LMS_Activity_Hours_Per_Week: 39 outliers
Pass_Chance_Percentage: 21 outliers


## Handle missing values

In [10]:
df[numerical_columns] = df[numerical_columns].fillna(
    df[numerical_columns].median()
)

In [11]:
df.isna().sum()

Gender                         0
Attendance_Percentage          0
Assignment_Average             0
Quiz_Average                   0
Midterm_Marks                  0
Study_Hours_Per_Week           0
Previous_Module_Marks          0
Class_Participation_Score      0
Missed_Assignments             0
LMS_Activity_Hours_Per_Week    0
Pass_Chance_Percentage         0
dtype: int64

## Remove outliers

In [12]:
# Print the number of rows and columns before removing outliers
print("Dataset before removing outliers:", df.shape)

# Remove outliers using the IQR (Interquartile Range) method
for column in numerical_columns:

    # Calculate Q1 (25th percentile) of the current numerical column
    Q1 = df[column].quantile(0.25)

    # Calculate Q3 (75th percentile) of the current numerical column
    Q3 = df[column].quantile(0.75)

    # Calculate the Interquartile Range (IQR)
    # IQR represents the middle 50% of the data
    IQR = Q3 - Q1

    # Calculate the lower boundary for detecting outliers
    # Values below this boundary are considered potential outliers
    lower_bound = Q1 - 1.5 * IQR

    # Calculate the upper boundary for detecting outliers
    # Values above this boundary are considered potential outliers
    upper_bound = Q3 + 1.5 * IQR

    # Keep only rows where the value is within the lower and upper boundaries
    df = df[
        (df[column] >= lower_bound) &  # Keep values greater than or equal to lower bound
        (df[column] <= upper_bound)    # Keep values less than or equal to upper bound
    ]

# Print the number of rows and columns after removing outliers
print("Dataset after removing outliers:", df.shape)

# Reset the row index after removing some rows
# drop=True prevents the old index from becoming a new column
# inplace=True applies the change directly to df
df.reset_index(drop=True, inplace=True)

# Display the first five rows of the cleaned dataset
df.head()

Dataset before removing outliers: (3000, 11)
Dataset after removing outliers: (2791, 11)


,Gender,Attendance_Percentage,Assignment_Average,Quiz_Average,Midterm_Marks,Study_Hours_Per_Week,Previous_Module_Marks,Class_Participation_Score,Missed_Assignments,LMS_Activity_Hours_Per_Week,Pass_Chance_Percentage
0,0,91.8,79.0,77.5,75.1,7.2,69.8,10.0,2,8.8,76.82
1,1,64.7,44.9,81.3,54.1,7.1,58.8,1.8,5,9.3,35.13
2,1,97.3,92.5,100.0,100.0,4.2,100.0,10.0,1,7.2,91.71
3,0,70.0,72.9,89.6,78.9,8.8,60.8,8.2,5,6.1,57.93
4,0,64.7,75.7,55.9,85.0,3.7,44.4,6.8,2,7.7,63.29


## Feuture Splitting

In [13]:
X = df.drop(['Pass_Chance_Percentage'], axis=1)
y = df['Pass_Chance_Percentage']
print(X)
print(y)

      Gender  Attendance_Percentage  Assignment_Average  Quiz_Average  \
0          0                   91.8                79.0          77.5   
1          1                   64.7                44.9          81.3   
2          1                   97.3                92.5         100.0   
3          0                   70.0                72.9          89.6   
4          0                   64.7                75.7          55.9   
...      ...                    ...                 ...           ...   
2786       1                   84.9                72.1          71.4   
2787       0                   78.6                63.7          59.2   
2788       1                   83.9                88.2          91.2   
2789       0                   65.6                47.7          52.7   
2790       1                   72.3                56.6          59.8   

      Midterm_Marks  Study_Hours_Per_Week  Previous_Module_Marks  \
0              75.1                   7.2              

## Split dataset

In [14]:
X_train, X_test, y_train,y_test=train_test_split(X, y, test_size=.2, random_state=42)
X_train.shape

(2232, 10)

In [15]:
X_test.shape

(559, 10)